# 01_train_study

- Ablation study for loss function selection
- Each run trains DeclipNet with the same architecture and a different loss function configuration
- All runs trained to convergence using early stopping with relaxed patience
  (fewer epochs without improvement than in final training)
- Tracks train loss and val SI-SDR per epoch — loss curve used to assess convergence
- Loss functions compared via paired comparisons on val SI-SDR at convergence
- All run results saved with run name, loss config, train loss history, and val SI-SDR history
- Winning loss function carried forward to 01_train_final for full training with stricter convergence

In [1]:
import sys
import os
import json
import time
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from pathlib import Path

sys.path.insert(0, "..")
from config import *
from util import (
    DeclipDataset,
    DeclipNet,
    l1_loss,
    weighted_l1_loss,
    multires_stft_loss,
    dwt_loss,
    si_sdr
)

In [2]:
# Hyperparameters
BATCH_SIZE = 32
LR = 1e-3
PATIENCE = 10        # epochs without improvement before early stopping (relaxed for study)
MIN_DELTA = 1e-3     # minimum SI-SDR improvement to count as progress
VAL_EVERY = 1        # evaluate val SI-SDR every N epochs

# Model
H = 8
N_ATTN = 3
NUM_HEADS = 4
FFN_DIM = 256

# Device
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: mps


In [3]:
# Dataset and dataloaders
train_dataset = DeclipDataset(TRAIN_OUT / "train_blocks.pt", TRAIN_OUT / "train_manifest.json")
val_dataset = DeclipDataset(VAL_OUT / "val_blocks.pt", VAL_OUT / "val_manifest.json")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples:   {len(val_dataset)}")

Train samples: 225045
Val samples:   28131


## Loss Function Ablation Study

### Stage 1a — Waveform loss: does amplitude weighting help?
| Run | Loss | Purpose |
|-----|------|---------|
| l1 | L1 | Baseline |
| weighted_l1 | Weighted L1 | Does amplitude weighting help? |

Winner carries forward as the waveform component in all subsequent stages.

### Stage 1b — Spectral loss: does it help, and which is better?
| Run | Loss | Purpose |
|-----|------|---------|
| winner + stft | Winner + STFT | Does spectral loss help? |
| winner + dwt | Winner + DWT | Is DWT better than STFT? |

### Stage 2 — Frequency weighting: does it help for the winning spectral term?
| Run | Loss | Purpose |
|-----|------|---------|
| winner + stft_weighted | Winner + weighted STFT | (only if STFT wins stage 1b) |
| winner + dwt_weighted | Winner + weighted DWT | (only if DWT wins stage 1b) |

### Interpretation
- Stage 1a winner = best waveform loss
- Stage 1b winner = best spectral term
- Stage 2 winner = final loss function for 01_train_final

In [4]:
# Define loss function options
def make_loss_fn(config_name):
    if config_name == "l1":
        return lambda out, tgt, clip: l1_loss(out, tgt)
    elif config_name == "weighted_l1":
        return lambda out, tgt, clip: weighted_l1_loss(out, tgt, clip, weight_power=1.0)
    elif config_name == "l1_stft":
        return lambda out, tgt, clip: l1_loss(out, tgt) + multires_stft_loss(out, tgt, weight_power=0.0)
    elif config_name == "l1_dwt":
        return lambda out, tgt, clip: l1_loss(out, tgt) + dwt_loss(out, tgt, weight_power=0.0)
    elif config_name == "l1_stft_weighted":
        return lambda out, tgt, clip: l1_loss(out, tgt) + multires_stft_loss(out, tgt, weight_power=1.0)
    elif config_name == "l1_dwt_weighted":
        return lambda out, tgt, clip: l1_loss(out, tgt) + dwt_loss(out, tgt, weight_power=1.0)
    elif config_name == "weighted_l1_stft":
        return lambda out, tgt, clip: weighted_l1_loss(out, tgt, clip, weight_power=1.0) + multires_stft_loss(out, tgt, weight_power=0.0)
    elif config_name == "weighted_l1_dwt":
        return lambda out, tgt, clip: weighted_l1_loss(out, tgt, clip, weight_power=1.0) + dwt_loss(out, tgt, weight_power=0.0)
    elif config_name == "weighted_l1_stft_weighted":
        return lambda out, tgt, clip: weighted_l1_loss(out, tgt, clip, weight_power=1.0) + multires_stft_loss(out, tgt, weight_power=1.0)
    elif config_name == "weighted_l1_dwt_weighted":
        return lambda out, tgt, clip: weighted_l1_loss(out, tgt, clip, weight_power=1.0) + dwt_loss(out, tgt, weight_power=1.0)
    else:
        raise ValueError(f"Unknown loss config: {config_name}")

In [5]:
def train_run(run_name, loss_fn, train_loader, val_loader, device,
              lr=LR, patience=PATIENCE, min_delta=MIN_DELTA):
    
    run_dir = STUDY_OUT / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    
    results_path = run_dir / "results.json"
    if results_path.exists():
        print(f"{run_name} already completed, skipping.")
        return
    
    torch.manual_seed(42)
    model = DeclipNet(H=H, N=N_ATTN, num_heads=NUM_HEADS, ffn_dim=FFN_DIM).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    train_loss_history = []
    val_sdr_history = []
    best_val_sdr = -float("inf")
    epochs_no_improve = 0
    epoch = 0
    
    while epochs_no_improve < patience:
        # Training
        model.train()
        epoch_loss = 0.0
        for clipped, clean in train_loader:
            clipped, clean = clipped.to(device), clean.to(device)
            optimizer.zero_grad()
            output = model(clipped)
            loss = loss_fn(output, clean, clipped)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        
        epoch_loss /= len(train_loader)
        train_loss_history.append(epoch_loss)
        
        # Validation
        model.eval()
        total_sdr = 0.0
        total_count = 0
        with torch.no_grad():
            for clipped, clean in val_loader:
                clipped, clean = clipped.to(device), clean.to(device)
                output = model(clipped)
                scores = si_sdr(clean, output)
                total_sdr += scores.sum().item()
                total_count += scores.numel()
        
        val_sdr = total_sdr / total_count
        val_sdr_history.append(val_sdr)
        
        # Early stopping
        if val_sdr > best_val_sdr + min_delta:
            best_val_sdr = val_sdr
            epochs_no_improve = 0
            torch.save(model.state_dict(), run_dir / "best_model.pt")
        else:
            epochs_no_improve += 1
        
        epoch += 1
        print(f"[{run_name}] epoch {epoch:03d} | loss: {epoch_loss:.6f} | val SI-SDR: {val_sdr:.4f} dB | best: {best_val_sdr:.4f} dB | no improve: {epochs_no_improve}/{patience}", flush=True)
    
    # Save results
    results = {
        "run_name": run_name,
        "best_val_sdr": best_val_sdr,
        "train_loss_history": train_loss_history,
        "val_sdr_history": val_sdr_history,
        "epochs": epoch,
    }
    with open(results_path, "w") as f:
        json.dump(results, f, indent=2)
    
    print(f"{run_name} complete. Best val SI-SDR: {best_val_sdr:.4f} dB")

In [6]:
import subprocess

# Stage 1a — waveform loss comparison
run_configs_1a = [
    {"name": "l1"},
    {"name": "weighted_l1"},
]

caffeinate = subprocess.Popen(["caffeinate", "-i"])
try:
    for config in run_configs_1a:
        train_run(
            run_name=config["name"],
            loss_fn=make_loss_fn(config["name"]),
            train_loader=train_loader,
            val_loader=val_loader,
            device=DEVICE,
        )
finally:
    caffeinate.terminate()
    print("Caffeinate terminated.")

[l1] epoch 001 | loss: 0.180489 | val SI-SDR: 14.5320 dB | best: 14.5320 dB | no improve: 0/10
[l1] epoch 002 | loss: 0.132932 | val SI-SDR: 14.8515 dB | best: 14.8515 dB | no improve: 0/10
[l1] epoch 003 | loss: 0.125566 | val SI-SDR: 15.5493 dB | best: 15.5493 dB | no improve: 0/10
[l1] epoch 004 | loss: 0.115250 | val SI-SDR: 16.2076 dB | best: 16.2076 dB | no improve: 0/10
[l1] epoch 005 | loss: 0.098476 | val SI-SDR: 16.8501 dB | best: 16.8501 dB | no improve: 0/10
[l1] epoch 006 | loss: 0.094404 | val SI-SDR: 17.1080 dB | best: 17.1080 dB | no improve: 0/10
[l1] epoch 007 | loss: 0.092062 | val SI-SDR: 17.0426 dB | best: 17.1080 dB | no improve: 1/10
[l1] epoch 008 | loss: 0.090458 | val SI-SDR: 16.9187 dB | best: 17.1080 dB | no improve: 2/10
[l1] epoch 009 | loss: 0.089355 | val SI-SDR: 17.6851 dB | best: 17.6851 dB | no improve: 0/10
[l1] epoch 010 | loss: 0.088065 | val SI-SDR: 17.7456 dB | best: 17.7456 dB | no improve: 0/10
[l1] epoch 011 | loss: 0.087632 | val SI-SDR: 17.5

## Stage 1a Results 